In [1]:
# ============================================================
# INSTALL REQUIRED PACKAGES
# ============================================================

!pip install -q kneed

In [2]:
# ============================================================
# IMPORTS
# ============================================================

import os
import json
import pickle
import warnings

import numpy as np
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

from PIL import Image

import torch
import torch.nn as nn

from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import convnext_tiny
from torch.utils.data import DataLoader

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from kneed import KneeLocator

warnings.filterwarnings("ignore")

print("All libraries imported successfully.")

All libraries imported successfully.


In [ ]:
# ============================================================
# DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

# ============================================================
# PATHS
# ============================================================

DATASET_PATH = (
    "/kaggle/input/datasets/sayan2807/uav-cropped-labeled-soyabean-leaf-image-dataset"
)

MODEL_PATH = "/kaggle/input/models/sayan2807/fine-tuned-conv-next-on-cropped-leaf-dataset/pytorch/default/1/uav_finetuned_convnext.pth"

OUTPUT_DIR = "/kaggle/working/prototype_database"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\nDataset :", DATASET_PATH)
print("Model   :", MODEL_PATH)
print("Output  :", OUTPUT_DIR)

Device : cuda

Dataset : /kaggle/input/datasets/sayan2807/uav-cropped-labeled-soyabean-leaf-image-dataset
Model   : /kaggle/input/models/sayan2807/fine-tuned-conv-next-on-cropped-leaf-dataset/pytorch/default/1/uav_finetuned_convnext.pth
Output  : /kaggle/working/prototype_database


In [ ]:
# ============================================================
# IMAGE TRANSFORM
# ============================================================

transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

print("Transform Created.")

Transform Created.


In [ ]:
# ============================================================
# LOAD DATASET
# ============================================================

dataset = ImageFolder(root=DATASET_PATH, transform=transform)

loader = DataLoader(
    dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True
)

CLASS_NAMES = dataset.classes

print("\nClasses Found:\n")

for i, cls in enumerate(CLASS_NAMES):
    print(f"{i} -> {cls}")

print("\nTotal Images :", len(dataset))


Classes Found:

0 -> Healthy
1 -> Mosaic
2 -> Rust
3 -> Semilooper

Total Images : 61478


In [ ]:
# ============================================================
# LOAD FINETUNED CONVNEXT
# ============================================================

print("\nLoading Fine-Tuned ConvNeXt...")

model = convnext_tiny(weights=None)

model.classifier[2] = nn.Linear(768, len(CLASS_NAMES))

checkpoint = torch.load(MODEL_PATH, map_location=device)

model.load_state_dict(checkpoint)

model.to(device)

model.eval()

print("ConvNeXt Loaded Successfully.")

# ============================================================
# FEATURE EXTRACTOR
# ============================================================

feature_extractor = nn.Sequential(model.features, model.avgpool, nn.Flatten())

feature_extractor.to(device)

feature_extractor.eval()

print("Feature Extractor Ready.")


Loading Fine-Tuned ConvNeXt...
ConvNeXt Loaded Successfully.
Feature Extractor Ready.


In [ ]:
# ============================================================
# FEATURE EXTRACTION
# ============================================================

print("\nExtracting Deep Features...\n")

all_features = []

all_labels = []

all_image_paths = []

progress_bar = tqdm(loader, desc="Extracting Features", unit="batch")

with torch.no_grad():
    for images, labels in progress_bar:
        images = images.to(device, non_blocking=True)

        features = feature_extractor(images)

        features = features.cpu().numpy()

        all_features.append(features)

        all_labels.extend(labels.numpy())

# ============================================================
# CONCATENATE FEATURES
# ============================================================

all_features = np.concatenate(all_features, axis=0)

all_labels = np.array(all_labels)

# ============================================================
# STORE IMAGE PATHS
# ============================================================

all_image_paths = [sample[0] for sample in dataset.samples]

print("\n===================================================")

print("FEATURE EXTRACTION COMPLETED")

print("===================================================\n")

print("Feature Shape :", all_features.shape)

print("Label Shape   :", all_labels.shape)

print("Images        :", len(all_image_paths))


Extracting Deep Features...



Extracting Features:   0%|          | 0/961 [00:00<?, ?batch/s]


FEATURE EXTRACTION COMPLETED

Feature Shape : (61478, 768)
Label Shape   : (61478,)
Images        : 61478


In [ ]:
# ============================================================
# ORGANIZE FEATURES CLASS-WISE
# ============================================================

class_feature_dict = {}

class_image_dict = {}

print("\nOrganizing Features...\n")

for class_index, class_name in enumerate(CLASS_NAMES):
    indices = np.where(all_labels == class_index)[0]

    class_feature_dict[class_name] = all_features[indices]

    class_image_dict[class_name] = [all_image_paths[i] for i in indices]

    print(f"{class_name:<35}{len(indices):>6} images")

print("\nDone.")


Organizing Features...

Healthy                              8381 images
Mosaic                              17110 images
Rust                                10283 images
Semilooper                          25704 images

Done.


In [ ]:
# ============================================================
# AUTOMATIC ELBOW METHOD
# ============================================================

print("\nFinding Optimal K using Elbow Method...\n")

optimal_k = {}

wcss_dict = {}

K_RANGE = range(2, 21)

progress_bar = tqdm(CLASS_NAMES, desc="Finding Optimal K", unit="class")

for class_name in progress_bar:
    features = class_feature_dict[class_name]

    wcss = []

    # ========================================================
    # COMPUTE WCSS
    # ========================================================

    for k in K_RANGE:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")

        kmeans.fit(features)

        wcss.append(kmeans.inertia_)

    wcss_dict[class_name] = wcss

    # ========================================================
    # AUTOMATIC ELBOW DETECTION
    # ========================================================

    knee = KneeLocator(list(K_RANGE), wcss, curve="convex", direction="decreasing")

    if knee.elbow is None:
        best_k = 10

    else:
        best_k = knee.elbow

    optimal_k[class_name] = int(best_k)

    progress_bar.set_postfix({"Current Class": class_name, "Optimal K": best_k})

print("\n===================================================")
print("ELBOW ANALYSIS COMPLETED")
print("===================================================\n")

for cls in CLASS_NAMES:
    print(f"{cls:<35}K = {optimal_k[cls]}")


Finding Optimal K using Elbow Method...



Finding Optimal K:   0%|          | 0/4 [00:00<?, ?class/s]


ELBOW ANALYSIS COMPLETED

Healthy                            K = 7
Mosaic                             K = 9
Rust                               K = 8
Semilooper                         K = 8


In [ ]:
# # ============================================================
# # GENERATE PROTOTYPES USING OPTIMAL K
# # ============================================================

# print("\nGenerating Prototype Database...\n")

# prototype_database = {}

# progress_bar = tqdm(

#     CLASS_NAMES,

#     desc="Generating Prototypes",

#     unit="class"
# )

# for class_name in progress_bar:

#     # ========================================================
#     # GET FEATURES
#     # ========================================================

#     features = class_feature_dict[class_name]

#     # ========================================================
#     # STANDARDIZE FEATURES
#     # ========================================================

#     scaler = StandardScaler()

#     scaled_features = scaler.fit_transform(
#         features
#     )

#     # ========================================================
#     # KMEANS
#     # ========================================================

#     k = optimal_k[class_name]

#     kmeans = KMeans(

#         n_clusters=k,

#         random_state=42,

#         n_init="auto"
#     )

#     cluster_labels = kmeans.fit_predict(
#         scaled_features
#     )

#     cluster_centers = kmeans.cluster_centers_

#     cluster_sizes = np.bincount(
#         cluster_labels
#     )

#     # ========================================================
#     # SAVE EVERYTHING
#     # ========================================================

#     prototype_database[class_name] = {

#     "k": k,

#     "centers": cluster_centers,

#     "cluster_sizes": cluster_sizes,

#     "feature_vectors": scaled_features,

#     "cluster_labels": cluster_labels,

#     "scaler": scaler,

#     "inertia": kmeans.inertia_
# }

# print("\n===================================================")
# print("PROTOTYPE GENERATION COMPLETED")
# print("===================================================\n")

# for cls in CLASS_NAMES:

#     print(

#         f"{cls:<35}"

#         f"Prototypes : "

#         f"{prototype_database[cls]['k']}"
#     )


from scipy.spatial.distance import cdist

print("\nGenerating Representative Prototype Database...\n")

prototype_database = {}

progress_bar = tqdm(CLASS_NAMES, desc="Generating Prototypes", unit="class")

for class_name in progress_bar:
    # -----------------------------------------
    # Features
    # -----------------------------------------

    features = class_feature_dict[class_name]

    scaler = StandardScaler()

    scaled_features = scaler.fit_transform(features)

    k = optimal_k[class_name]

    # -----------------------------------------
    # KMeans
    # -----------------------------------------

    kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")

    cluster_labels = kmeans.fit_predict(scaled_features)

    centers = kmeans.cluster_centers_

    representative_vectors = []
    representative_indices = []
    cluster_sizes = []

    # -----------------------------------------
    # Representative prototype of each cluster
    # -----------------------------------------

    for cluster_id in range(k):
        idx = np.where(cluster_labels == cluster_id)[0]

        cluster_features = scaled_features[idx]

        cluster_sizes.append(len(idx))

        distances = cdist(
            cluster_features, centers[cluster_id].reshape(1, -1), metric="euclidean"
        )

        nearest = np.argmin(distances)

        representative_vectors.append(cluster_features[nearest])

        representative_indices.append(idx[nearest])

    representative_vectors = np.array(representative_vectors)

    prototype_database[class_name] = {
        "k": k,
        "prototype_vectors": representative_vectors,
        "prototype_indices": representative_indices,
        "cluster_sizes": np.array(cluster_sizes),
        "feature_vectors": scaled_features,
        "cluster_labels": cluster_labels,
        "scaler": scaler,
        "inertia": kmeans.inertia_,
    }

print("\n===========================================")
print("REPRESENTATIVE PROTOTYPE DATABASE CREATED")
print("===========================================\n")

for cls in CLASS_NAMES:
    print(f"{cls:<35}{prototype_database[cls]['k']} prototypes")


Generating Representative Prototype Database...



Generating Prototypes:   0%|          | 0/4 [00:00<?, ?class/s]


REPRESENTATIVE PROTOTYPE DATABASE CREATED

Healthy                            7 prototypes
Mosaic                             9 prototypes
Rust                               8 prototypes
Semilooper                         8 prototypes


In [ ]:
# ============================================================
# SAVE PROTOTYPE DATABASE
# ============================================================

prototype_path = os.path.join(OUTPUT_DIR, "prototypes.pkl")

with open(prototype_path, "wb") as f:
    pickle.dump(prototype_database, f)

print("\nPrototype Database Saved.")

print(prototype_path)

# ============================================================
# SAVE OPTIMAL K
# ============================================================

k_path = os.path.join(OUTPUT_DIR, "optimal_k.json")

with open(k_path, "w") as f:
    json.dump(optimal_k, f, indent=4)

print("\nOptimal K Saved.")

print(k_path)


Prototype Database Saved.
/kaggle/working/prototype_database/prototypes.pkl

Optimal K Saved.
/kaggle/working/prototype_database/optimal_k.json


In [ ]:
# ============================================================
# SAVE PROTOTYPE DATABASE
# ============================================================

import pickle
import json
import os

prototype_file = os.path.join(OUTPUT_DIR, "prototype_database.pkl")

with open(prototype_file, "wb") as f:
    pickle.dump(prototype_database, f)

print("\nPrototype database saved successfully.")

print(prototype_file)

# ============================================================
# SAVE OPTIMAL K
# ============================================================

k_file = os.path.join(OUTPUT_DIR, "optimal_k.json")

with open(k_file, "w") as f:
    json.dump(optimal_k, f, indent=4)

print("\nOptimal K saved.")

print(k_file)


Prototype database saved successfully.
/kaggle/working/prototype_database/prototype_database.pkl

Optimal K saved.
/kaggle/working/prototype_database/optimal_k.json


In [15]:
print(prototype_database.keys())

print()

print(prototype_database["Healthy"].keys())

dict_keys(['Healthy', 'Mosaic', 'Rust', 'Semilooper'])

dict_keys(['k', 'prototype_vectors', 'prototype_indices', 'cluster_sizes', 'feature_vectors', 'cluster_labels', 'scaler', 'inertia'])


In [ ]:
# ============================================================
# PROTOTYPE SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("PROTOTYPE DATABASE SUMMARY")
print("=" * 70)

total_prototypes = 0

for cls in CLASS_NAMES:
    k = prototype_database[cls]["k"]

    total_prototypes += k

    print(f"\nClass : {cls}")

    print(f"Optimal K        : {k}")

    print(f"Cluster Sizes    : {prototype_database[cls]['cluster_sizes']}")

    print(f"Feature Shape    : {prototype_database[cls]['feature_vectors'].shape}")

    print(f"Prototype Shape  : {prototype_database[cls]['centers'].shape}")

print("\n")
print("=" * 70)
print(f"TOTAL PROTOTYPES : {total_prototypes}")
print("=" * 70)



PROTOTYPE DATABASE SUMMARY

Class : Healthy
Optimal K        : 7
Cluster Sizes    : [1228 1282  961 1579  626 1428 1277]
Feature Shape    : (8381, 768)


KeyError: 'centers'

In [ ]:
# ============================================================
# PROTOTYPE SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("PROTOTYPE DATABASE SUMMARY")
print("=" * 70)

total_prototypes = 0

for cls in CLASS_NAMES:
    k = prototype_database[cls]["k"]

    total_prototypes += k

    print(f"\nClass : {cls}")

    print(f"Optimal K        : {k}")

    print(f"Cluster Sizes    : {prototype_database[cls]['cluster_sizes']}")

    print(f"Feature Shape    : {prototype_database[cls]['feature_vectors'].shape}")

    print(f"Prototype Shape  : {prototype_database[cls]['centers'].shape}")

print("\n")
print("=" * 70)
print(f"TOTAL PROTOTYPES : {total_prototypes}")
print("=" * 70)

In [ ]:
# ============================================================
# PROTOTYPE DISTRIBUTION
# ============================================================

import matplotlib.pyplot as plt

prototype_counts = [prototype_database[cls]["k"] for cls in CLASS_NAMES]

plt.figure(figsize=(8, 5))

plt.bar(CLASS_NAMES, prototype_counts)

plt.title("Number of Prototypes per Disease Class")

plt.xlabel("Disease Class")

plt.ylabel("Number of Prototypes")

plt.xticks(rotation=20)

plt.grid(axis="y")

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# ZIP PROTOTYPE DATABASE
# ============================================================

import os
import shutil

SOURCE_FOLDER = "/kaggle/working/prototype_database"

ZIP_NAME = "/kaggle/working/prototype_database"

shutil.make_archive(base_name=ZIP_NAME, format="zip", root_dir=SOURCE_FOLDER)

print("=" * 60)
print("ZIP FILE CREATED SUCCESSFULLY")
print("=" * 60)
print(f"Location : {ZIP_NAME}.zip")